In [ ]:
# Celula unica - Importacao de dados GPKG para PostgreSQL
# Aulas 07, 08 e 09 - GEOTEC
import geopandas as gpd
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# =============
# CONFIGURACOES 
# =============

# Pasta onde estao os arquivos GPKG
PASTA_DADOS = Path(r"C:\Temp")

# =============
# ARQUIVOS POR AULA
# =============

# Aula 07 - CAR MT e SIGEF
CAR_MT = "es_mt_car_20260406.gpkg"
SIGEF = "pa_br_sigef_privado_20260107.gpkg"

# Aula 08 - CAR PI 
CAR_PI = "es_pi_car_20260406.gpkg"

# Aula 09 - CAR MT (multiplas versoes) e Modulo Fiscal
CAR_MT_VERSOES = [
    "es_mt_car_20250702.gpkg",
    "es_mt_car_20251202.gpkg",
    "es_mt_car_20260406.gpkg"
]

# Aula 08 - CAR PI - Modulo Fiscal 
MODULO_FISCAL = "pa_br_modulosfiscais_incra.gpkg"

# =============
# CONTROLE DE IMPORTACAO
# =============
IMPORTAR_CAR_MT = True          # Aula 07
IMPORTAR_CAR_PI = True          # Aula 08
IMPORTAR_CAR_MT_VERSOES = True  # Aula 09
IMPORTAR_SIGEF = True           # Aulas 07 e 08
IMPORTAR_MODULO_FISCAL = True   # Aulas 08 e 09

# Controle geral
RECRIAR_TABELAS = False  # TRUE = sempre sobrescreve, FALSE = mantém existente

# Credenciais do banco
DB_HOST = "localhost"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_NAME = "geotec"

# ==========================================
# FUNCOES
# ==========================================

def conectar_banco():
    return create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

def criar_schemas(engine):
    with engine.connect() as conn:
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS car"))
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS incra"))
        conn.execute(text("CREATE EXTENSION IF NOT EXISTS postgis"))
        conn.commit()
    print("[OK] Schemas car e incra prontos")

def verificar_tabela(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            conn.execute(text(f"SELECT 1 FROM {schema}.{tabela} LIMIT 1"))
            return True
    except:
        return False

def contar_registros(engine, schema, tabela):
    try:
        with engine.connect() as conn:
            result = conn.execute(text(f"SELECT COUNT(*) FROM {schema}.{tabela}"))
            return result.fetchone()[0]
    except:
        return 0

def importar_gpkg(engine, caminho_arquivo, schema, tabela):
    print(f"  Lendo: {caminho_arquivo.name}")
    gdf = gpd.read_file(str(caminho_arquivo))
    print(f"  Registros: {len(gdf)}")

    if 'geometry' in gdf.columns:
        if gdf.crs is None:
            gdf = gdf.set_crs(epsg=4674)
        else:
            gdf = gdf.to_crs(epsg=4674)

        gdf.to_postgis(
            name=tabela,
            con=engine,
            schema=schema,
            if_exists='replace',
            index=False
        )
    else:
        df = pd.DataFrame(gdf)
        df.to_sql(
            name=tabela,
            con=engine,
            schema=schema,
            if_exists='replace',
            index=False
        )

    print(f"  OK → {schema}.{tabela}")
    return len(gdf)

def importar_se_necessario(engine, caminho, schema, tabela, ativo):
    if not ativo:
        return
    if not caminho.exists():
        print(f"  [ERRO] Arquivo nao encontrado: {caminho.name}")
        return
    if verificar_tabela(engine, schema, tabela) and not RECRIAR_TABELAS:
        print(f"  Mantido (ja existe): {schema}.{tabela}")
    else:
        importar_gpkg(engine, caminho, schema, tabela)

# ==========================================
# MAIN
# ==========================================

def main():
    print("=" * 60)
    print("IMPORTACAO DE DADOS - AULAS 07, 08 e 09")
    print("=" * 60)

    if not PASTA_DADOS.exists():
        print(f"[ERRO] Pasta nao encontrada: {PASTA_DADOS}")
        return

    engine = conectar_banco()
    print("[OK] Conectado ao PostgreSQL")

    criar_schemas(engine)

    # =====================
    # AULA 07 - CAR MT
    # =====================
    print("\n[1] AULA 07 - CAR MT")
    caminho_car_mt = PASTA_DADOS / CAR_MT
    importar_se_necessario(engine, caminho_car_mt, "car", "es_mt_car_20260406", IMPORTAR_CAR_MT)

    # =====================
    # AULA 08 - CAR PI
    # =====================
    print("\n[2] AULA 08 - CAR PI")
    caminho_car_pi = PASTA_DADOS / CAR_PI
    importar_se_necessario(engine, caminho_car_pi, "car", "es_pi_car_20260406", IMPORTAR_CAR_PI)

    # =====================
    # AULA 09 - CAR MT (multiplas versoes)
    # =====================
    if IMPORTAR_CAR_MT_VERSOES:
        print("\n[3] AULA 09 - CAR MT (multiplas versoes)")
        for arquivo in CAR_MT_VERSOES:
            caminho = PASTA_DADOS / arquivo
            tabela = Path(arquivo).stem
            importar_se_necessario(engine, caminho, "car", tabela, IMPORTAR_CAR_MT_VERSOES)
    else:
        print("\n[3] AULA 09 - CAR MT (multiplas versoes) desativado")

    # =====================
    # SIGEF (Aulas 07 e 08)
    # =====================
    print("\n[4] SIGEF")
    caminho_sigef = PASTA_DADOS / SIGEF
    importar_se_necessario(engine, caminho_sigef, "incra", "pa_br_sigef_privado_20260107", IMPORTAR_SIGEF)

    # =====================
    # MODULO FISCAL (Aula 09)
    # =====================
    print("\n[5] MODULO FISCAL")
    caminho_mod_fiscal = PASTA_DADOS / MODULO_FISCAL
    importar_se_necessario(engine, caminho_mod_fiscal, "incra", "pa_br_modulosfiscais_incra", IMPORTAR_MODULO_FISCAL)

    # =====================
    # RESUMO FINAL
    # =====================
    print("\n" + "=" * 60)
    print("RESUMO DAS BASES IMPORTADAS")
    print("=" * 60)

    # CAR MT (Aula 07)
    print(f"\n  Aula 07 - CAR MT:")
    print(f"    car.es_mt_car_20260406: {contar_registros(engine, 'car', 'es_mt_car_20260406')} registros")

    # CAR PI (Aula 08)
    print(f"\n  Aula 08 - CAR PI:")
    print(f"    car.es_pi_car_20260406: {contar_registros(engine, 'car', 'es_pi_car_20260406')} registros")

    # CAR MT Versoes (Aula 09)
    print(f"\n  Aula 09 - CAR MT (versoes):")
    for arquivo in CAR_MT_VERSOES:
        tabela = Path(arquivo).stem
        print(f"    car.{tabela}: {contar_registros(engine, 'car', tabela)} registros")

    # SIGEF
    print(f"\n  SIGEF:")
    print(f"    incra.pa_br_sigef_privado_20260107: {contar_registros(engine, 'incra', 'pa_br_sigef_privado_20260107')} registros")

    # Modulo Fiscal
    print(f"\n  Modulo Fiscal:")
    print(f"    incra.pa_br_modulosfiscais_incra: {contar_registros(engine, 'incra', 'pa_br_modulosfiscais_incra')} registros")

    engine.dispose()

    print("\n" + "=" * 60)
    print("PRONTO! BASES IMPORTADAS COM SUCESSO")
    print("=" * 60)

if __name__ == "__main__":
    main()